# Track-A v1.2 — MASTER K3 (v6)

Two-GPU worker. Acquires canonical K1 G1A, runs CAL-R13 on GPU0 during G2A, waits for the sealed control plane, then uses both GPUs during K3 science.

Use **Kaggle T4 x2**, **Internet ON**, and **Save Version -> Save & Run All / Batch**.

**Frozen science SHA:** `9a72e9466a9a3e7429e0e36a028edac662f83146`  
**Pinned operator runtime:** `208656f895af5c218a0998b582f2adbb81167eaa`  
**Runtime branch:** `ops-tracka-kaggle-master-runtime-v6-208656f`

Attach the frozen CropCop V1 dataset. The canonical K1 private G1A dataset must be shared with this K3 Kaggle account before handoff acquisition can succeed.

Required Kaggle secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `CROPCOP_GITHUB_TOKEN`.

v6 uses a process-isolated runtime checkout and a per-account singleton so accidental overlapping K3 launches cannot race dependency, G2A, control, or science state.

Do not edit scientific settings in this notebook.


In [ ]:
from pathlib import Path
import os
V1 = Path('/kaggle/input/datasets/ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1/CropCop_Final_v1')
if V1.is_dir():
    preferred = {
        'CROPCOP_MANIFEST': V1 / 'audit' / 'final_manifest.csv',
        'CROPCOP_CLASS_MAP': V1 / 'audit' / 'class_to_idx.json',
        'CROPCOP_IMAGE_ROOT': V1 / 'dataset',
    }
    for name, path in preferred.items():
        if path.exists():
            os.environ.setdefault(name, str(path))
    print('Preferred frozen V1 mount detected:', V1)
else:
    print('Preferred V1 mount prefix not present; bounded hash/structure resolver will be used.')
# Optional fail-closed account binding.
# os.environ['CROPCOP_EXPECTED_KAGGLE_USERNAME'] = 'your-k3-kaggle-username'


In [ ]:
from pathlib import Path
import os, subprocess, sys, uuid
OPS_RUNTIME_SHA = '208656f895af5c218a0998b582f2adbb81167eaa'
OPS_RUNTIME_BRANCH = 'ops-tracka-kaggle-master-runtime-v6-208656f'
OPS_ROOT = Path(f'/kaggle/working/cropcop-tracka-master-runtime-K3-{os.getpid()}-{uuid.uuid4().hex[:8]}')
subprocess.run([
    'git', 'clone', '--quiet', '--depth', '1', '--branch', OPS_RUNTIME_BRANCH,
    'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git', str(OPS_ROOT)
], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=OPS_ROOT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=OPS_ROOT, text=True).strip()
if head != OPS_RUNTIME_SHA:
    raise RuntimeError(f'operator runtime SHA mismatch: expected {OPS_RUNTIME_SHA}, got {head}')
if dirty:
    raise RuntimeError(f'operator runtime checkout is dirty: {dirty}')
print('Pinned Track-A master runtime:', head)
print('Process-isolated runtime checkout:', OPS_ROOT)


In [ ]:
driver = OPS_ROOT / 'journal_extension/kaggle/tracka_v12_ops/master_account_driver_v6.py'
cp = subprocess.run([sys.executable, '-u', str(driver), 'K3'], cwd=driver.parent)
if cp.returncode == 0:
    print('K3 master: TERMINAL PASS for this account queue.')
elif cp.returncode == 2:
    print('K3 master: controlled dependency/session/publication continuation. Rerun THIS SAME notebook as a fresh Batch version.')
else:
    raise RuntimeError(f'K3 master requires investigation; rc={cp.returncode}')


Recovery rule: use this same K3 notebook in a **fresh Batch session** for controlled continuation. Same-session duplicate execution is serialized and mirrored, not rerun.
